# Анализ датасета дата-центров по странам мира


#### __1. Формулировка бизнес-задачи__

__Бизнес-задача:__ Международная компания планирует расширение своей инфраструктуры дата-центров и нуждается в анализе текущего рынка дата-центров по странам для определения наиболее перспективных локаций для инвестиций.

__Цель анализа:__ Выявить ключевые факторы, влияющие на развитие дата-центров в разных странах, и определить наиболее привлекательные рынки для инвестирования.

#### __2. Гипотезы о факторах, влияющих на решение задачи__

- **Гипотеза 1:** Страны с высоким уровнем интернет-проникновения имеют больше дата-центров
- **Гипотеза 2:** Использование возобновляемой энергии положительно коррелирует с количеством дата-центров
- **Гипотеза 3:** Низкая задержка (latency) до глобальных хабов способствует развитию дата-центров
- **Гипотеза 4:** Наличие крупных облачных провайдеров стимулирует строительство дата-центров
- **Гипотеза 5:** Страны с большей площадью дата-центров имеют большую мощность

#### __3. Определение необходимых и доступных данных__
 
 __Необходимые данные:__
 - Количество дата-центров по странам
 - Мощность дата-центров
 - Использование возобновляемой энергии
 - Интернет-проникновение
 - Задержка до глобальных хабов
 - Наличие облачных провайдеров
 
 __Доступные данные:__ Используется датасет Book1.xlsx, содержащий данные о дата-центрах по странам мира, включая показатели мощности, энергопотребления, интернет-инфраструктуры и наличия облачных провайдеров.
 
##### __Номинальная шкала:__
- country (Название страны)
- key_operators (Ключевые операторы дата-центров)
- cloud_provider (Наличие облачных провайдеров)
- cooling_technologies_common (Технологии охлаждения)
- regulatory_challenges_or_limits (Регуляторные ограничения)
- green_dc_initiatives_description (Описание "зеленых" инициатив)

##### __Шкала отношений:__

###### __Исходные:__
- total_data_centers (Общее количество дата-центров)
- hyperscale_data_centers (Количество гиперскейл-дата-центров)
- colocation_data_centers (Количество колокейшн-дата-центров)
- floor_space_sqft_total (Общая площадь в кв. футах)
- power_capacity_MW_total (Общая мощность в МВт)
- average_renewable_energy_usage_percent (Средний процент использования возобновляемой энергии)
- internet_penetration_percent (Процент интернет-проникновения)
- avg_latency_to_global_hubs_ms (Средняя задержка до глобальных хабов в мс)
- number_of_fiber_connections (Количество оптоволоконных соединений)
- growth_rate_of_data_centers_percent_per_year (Темп роста дата-центров в процентах в год)

###### __Производные (после вычислений):__
- tier_3_percentage (Процент дата-центров Tier III уровня)
- total_cloud_providers (Общее количество облачных провайдеров)
- has_aws / has_azure / has_gcp (Наличие конкретных облачных провайдеров)
- cooling_air / cooling_liquid / cooling_immersion / cooling_free (Наличие конкретных технологий охлаждения)

###### __Трансформированные:__
- total_data_centers_std (стандартизированное количество дата-центров)
- power_capacity_MW_norm (нормализованная мощность)
- renewable_energy_percent_std (стандартизированный процент возобновляемой энергии)

#### __4. Загрузка и подготовка набора данных__

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
import re
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Загрузка данных
df = pd.read_csv('Data\Dataset.csv', sep = ',', encoding='latin1')

In [56]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df.shape)

print("\nПропуски в данных:")
print(df.isnull().sum())


Размер датасета: (191, 20)

Пропуски в данных:
country                                           0
total_data_centers                                0
hyperscale_data_centers                           0
colocation_data_centers                           0
floor_space_sqft_total                            0
power_capacity_MW_total                           0
average_renewable_energy_usage_percent            0
key_operators                                    21
cloud_provider                                   42
internet_penetration_percent                      0
avg_latency_to_global_hubs_ms                     0
number_of_fiber_connections                       0
growth_rate_of_data_centers_percent_per_year      2
cooling_technologies_common                       1
regulatory_challenges_or_limits                   0
disaster_recovery_sites_common                   13
green_dc_initiatives_description                 26
source_of_data                                    4
Unnamed: 18     

In [57]:
# Проверка дубликатов
print(f"Количество дубликатов: {df.duplicated().sum()}")

Количество дубликатов: 0


In [58]:
# осмотр типов данных
print(df.dtypes)

country                                         object
total_data_centers                               int64
hyperscale_data_centers                         object
colocation_data_centers                         object
floor_space_sqft_total                          object
power_capacity_MW_total                         object
average_renewable_energy_usage_percent          object
key_operators                                   object
cloud_provider                                  object
internet_penetration_percent                    object
avg_latency_to_global_hubs_ms                   object
number_of_fiber_connections                     object
growth_rate_of_data_centers_percent_per_year    object
cooling_technologies_common                     object
regulatory_challenges_or_limits                 object
disaster_recovery_sites_common                  object
green_dc_initiatives_description                object
source_of_data                                  object
Unnamed: 1

#### __5. Очистка данных__

In [59]:
# копия для очистки
df_clean = df.copy()
df_clean = df_clean.sort_values('total_data_centers', ascending=False)

In [60]:
# удаление дубликатов
df_clean.drop_duplicates()

,country,total_data_centers,hyperscale_data_centers,colocation_data_centers,floor_space_sqft_total,power_capacity_MW_total,average_renewable_energy_usage_percent,key_operators,cloud_provider,internet_penetration_percent,avg_latency_to_global_hubs_ms,number_of_fiber_connections,growth_rate_of_data_centers_percent_per_year,cooling_technologies_common,regulatory_challenges_or_limits,disaster_recovery_sites_common,green_dc_initiatives_description,source_of_data,Unnamed: 18,Unnamed: 19
0,United States,5426,300+,"~2,000+","~3,552,000,000","~12,000+",~27%,"Equinix, Digital Realty, CoreSite, AWS","AWS:Yes, GCP:Yes, Azure:Yes, Oracle:Yes",92%,40,60+,12%,"Air, Liquid, Immersion","Power grid strain in VA, moratoriums in some s...","Yes, especially for finance/health","Solar, wind PPA, immersion cooling, LEED","Statista, Brightlio, IEA, CBRE",NaN,NaN
1,Germany,529,50,~200,"~350,000,000","~2,000+",40%+,"Equinix, Digital Realty, e-shelter","AWS:Yes, GCP:Yes, Azure:Yes, Oracle:Yes",92%,50,30+,8%,"Air, Indirect Evap.","Power caps in Frankfurt, strict emissions","Yes, especially for finance","Wind, solar, carbon-neutral push","Statista, CBRE, IEA",NaN,NaN
2,United Kingdom,523,30,~150,"~200,000,000","~1,000+",0.45,"Equinix, Telehouse, Virtus","AWS:Yes, GCP:Yes, Azure:Yes, Oracle:Yes",96%,40,25+,7%,"Air, Free Cooling",Power grid constraints in London,"Yes, especially for finance","Wind, solar, carbon-neutral targets","Statista, CBRE, IEA",NaN,NaN
3,China,449,100,"~1,000+","~500,000,000","~8,000+",0.25,"China Unicom, Tencent, Alibaba, GDS","AWS:Yes, GCP:No, Azure:Yes, Oracle:Yes",76%,75,40+,15%,"Air, Liquid",Power rationing in some provinces,"Yes, for government/finance","Hydro, wind, solar, green finance","Statista, CBRE, IEA",NaN,NaN
5,Canada,336,15,~100,"~90,000,000",~500,0.78,"Equinix, Cologix, eStruxture","AWS:Yes, GCP:Yes, Azure:Yes, Oracle:Yes",96%,75,18+,9%,"Air, Liquid, Free Cooling",Power grid constraints in Toronto/Vancouver,"Yes, especially for finance/health","Hydro, wind, carbon-neutral push","Statista, CBRE, IEA",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Guyana,0,0,0,0,0,37.00,"GTT, Digicel, ENet",No hyperscale presence,37%,200,3,1500%,"Air cooling, small precision AC, satellite backup","Power stability, remote connectivity challenges",1 (upcoming second DR site),Deep Blue,NaN,NaN,NaN
84,Guinea-Bissau,0,0,0,0,0,32.50,"MTN, Orange, Guinetel",No hyperscale presence,32%,200,1â2,300%,"Air cooling, small precision AC","Power instability, regulatory rebuilding",Few formal DR sites,No green DC initiatives yet,"DataReportal 2025, Internet Society Pulse, WAT...",NaN,NaN
81,Grenada,0,0,0,0,0,74.00,"Digicel, Flow, CCGL",No hyperscale presence,74%,150,2â3,500%,"Air cooling, small precision AC",Power reliability & limited redundancy,Few formal DR sites,"GREX IXP; basic connectivity, minimal green in...",DataReportal 2025; Internet Society Pulse; Tel...,NaN,NaN
155,Sao Tome and Principe,0,0,0,0,0,61.50,NaN,No global hyperscale presence,61.5,30,1,200%,"Satellite/VSAT, mobile 3G/4G, basic UPS","Poor ISP competition, limited caching (42%), i...","Grid cost, affordability, rural gaps, reliance...",0,"ACE submarine cable (2012), 1 IXP, <1% IPv6, S...","DataReportal, Internet Society Pulse, DigWatch...",NaN


In [61]:
# Удалить непонятные 2 последних столбца
cols_to_drop = ['Unnamed: 18', 'Unnamed: 19']  # или конкретные названия
df_clean = df_clean.drop(cols_to_drop, axis=1, errors='ignore')

In [62]:
# Функция для очистки строк с числами
def clean_numeric(value):
    if pd.isna(value): return np.nan
      
    str_value = str(value).strip()
    
    cleaned = re.sub(r'[^0-9.\-]', '', str(value).replace(',', ''))
    
    try:
        return float (cleaned)
    except:
        return np.nan
    
# Список колонок для преобразования в числовой формат
numbers_cols = [
    'total_data_centers', 'hyperscale_data_centers', 'colocation_data_centers',
    'floor_space_sqft_total', 'power_capacity_MW_total', 'avg_latency_to_global_hubs_ms', 
    'number_of_fiber_connections'
]

# Очистка числовых колонок
for col in numbers_cols:
    df_clean[col] = df_clean[col].apply(clean_numeric)
    
df_clean = df_clean.dropna (subset = numbers_cols)

print("Типы данных после чистки:")
print(df_clean[numbers_cols].dtypes)

Типы данных после чистки:
total_data_centers               float64
hyperscale_data_centers          float64
colocation_data_centers          float64
floor_space_sqft_total           float64
power_capacity_MW_total          float64
avg_latency_to_global_hubs_ms    float64
number_of_fiber_connections      float64
dtype: object


In [63]:
# Функция для очистки процентов
def clean_percent(value):
    if pd.isna(value): 
        return np.nan
    
    str_value = str(value).strip()
    cleaned = re.sub(r'[^0-9.\-]', '', str_value)
    
    try:
        num_value = float(cleaned)
        if num_value > 1:
            num_value = num_value / 100
        return round(num_value, 2)  # Округление здесь
    except:
        return np.nan
    
# Список колонок с процентами
percent_cols = [
    'average_renewable_energy_usage_percent',
    'growth_rate_of_data_centers_percent_per_year',
    'internet_penetration_percent'
]

# Очистка числовых колонок
for col in percent_cols:
    df_clean[col] = df_clean[col].apply(clean_percent)
    
df_clean = df_clean.dropna (subset = percent_cols)

print("Типы данных после чистки:")
print(df_clean[percent_cols].dtypes)

Типы данных после чистки:
average_renewable_energy_usage_percent          float64
growth_rate_of_data_centers_percent_per_year    float64
internet_penetration_percent                    float64
dtype: object


In [66]:
# Обработка cloud_provider - бинарные признаки
def has_cloud_provider(provider_str, provider_name):
    if pd.isna(provider_str):
        return 0
    if provider_name in str(provider_str) and 'Yes' in str(provider_str):
        return 1
    return 0

df_clean['has_aws'] = df_clean['cloud_provider'].apply(lambda x: has_cloud_provider(x, 'AWS'))
df_clean['has_azure'] = df_clean['cloud_provider'].apply(lambda x: has_cloud_provider(x, 'Azure'))
df_clean['has_gcp'] = df_clean['cloud_provider'].apply(lambda x: has_cloud_provider(x, 'GCP'))

# Признак общего количества облачных провайдеров
df_clean['total_cloud_providers'] = df_clean['has_aws'] + df_clean['has_azure'] + df_clean['has_gcp']

df_clean = df_clean.drop("cloud_provider", axis=1, errors='ignore')

In [64]:
# размер датасета и наличие пропусков
print("\nРазмер датасета:", df_clean.shape)

print("\nПропуски в данных:")
print(df_clean.isnull().sum())


Размер датасета: (129, 18)

Пропуски в данных:
country                                          0
total_data_centers                               0
hyperscale_data_centers                          0
colocation_data_centers                          0
floor_space_sqft_total                           0
power_capacity_MW_total                          0
average_renewable_energy_usage_percent           0
key_operators                                   17
cloud_provider                                   1
internet_penetration_percent                     0
avg_latency_to_global_hubs_ms                    0
number_of_fiber_connections                      0
growth_rate_of_data_centers_percent_per_year     0
cooling_technologies_common                      1
regulatory_challenges_or_limits                  0
disaster_recovery_sites_common                  12
green_dc_initiatives_description                22
source_of_data                                   2
dtype: int64


#### __5. Определение шкалы измерения признаков__

In [67]:
df_clean.to_csv('Data\Dataset_cleaned.csv', index = False, encoding='utf-8')